# Phase 3: Business Name Feature Analysis

This notebook benchmarks string similarity metrics (RapidFuzz, Jaccard, structural) on business names, explicitly tying back to Phase 1 EDA findings.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent.parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rapidfuzz import fuzz

from src.features.name_features import extract_name_features, _clean_str

sns.set_theme(style="whitegrid")

## 1. Create Difficult Benchmark Pairs
We create a dataset containing positive matches (typos, abbreviations, word reordering) and negative non-matches (similar industry but different company, different locations).

In [ ]:
pairs = [
    # Positive Pairs (Label = 1)
    ("abc technologies private limited", "abc technologies pvt ltd", "private limited", "private limited", 1), # Abbrev
    ("apple inc", "apple incorporated", "inc", "inc", 1),
    ("corporation of chennai", "chennai corporation", None, None, 1), # Reordering
    ("williams and sons", "williams & sons", None, None, 1), # Ampersand
    ("global tech", "global tehc", None, None, 1), # Typo
    
    # Negative Pairs (Label = 0)
    ("apple inc", "banana inc", "inc", "inc", 0), # Same suffix, different name
    ("tata motors", "tata steel", None, None, 0), # Same parent, different entity
    ("national bank", "national", None, None, 0), # Subset
    ("shree ram enterprises", "shree krishna enterprises", None, None, 0),
    
    # Edge Cases
    ("", "missing", None, None, 0),
    (None, None, None, None, 0)
]

df = pd.DataFrame(pairs, columns=["name1", "name2", "suffix1", "suffix2", "label"])
df

## 2. Extract Features using `name_features.py`

In [ ]:
features_list = []
for _, row in df.iterrows():
    feats = extract_name_features(
        row['name1'], 
        row['name2'], 
        row['suffix1'], 
        row['suffix2']
    )
    features_list.append(feats)

feat_df = pd.DataFrame(features_list)
result_df = pd.concat([df, feat_df], axis=1)
result_df

## 3. Visualize Feature Separation
Checking if our chosen features provide good separation between matches and non-matches.

In [ ]:
metrics = ["name_ratio", "name_token_sort", "name_norm_edit", "name_jaccard", "name_len_ratio"]

fig, axes = plt.subplots(1, len(metrics), figsize=(20, 4))
for i, metric in enumerate(metrics):
    sns.boxplot(data=result_df, x="label", y=metric, ax=axes[i])
    axes[i].set_title(f"Distribution of {metric}")
    axes[i].set_ylim(-0.1, 1.1)

plt.tight_layout()
plt.show()

## 4. Why we rejected Partial Ratio
Partial ratio matches the best substring. Look at how it fails on generic subsets:

In [ ]:
print("Partial Ratio (National Bank vs National):", fuzz.partial_ratio("national bank", "national"))
print("Token Sort Ratio (National Bank vs National):", fuzz.token_sort_ratio("national bank", "national"))
print("\nPartial ratio gives 100% to non-matching entities simply because one word is contained within the other. Token Sort is safer.")